# Ch 18 — 어텐션과 로짓 들여다보기

head 별 attention map 시각화와 top-k logit 추적으로 모델 내부 신호를 읽습니다.

In [ ]:
# 필요 시 설치
# !pip install torch matplotlib

import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 100

device = 'cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu'
print(f'device: {device}')

## 1. 개념 — 모델 내부의 두 가지 신호

| 신호 | 무엇 | 무엇을 답함 |
|---|---|---|
| **Attention map** | 각 head 의 (T, T) softmax 행렬 | "토큰 i 가 j 를 얼마나 보나" |
| **Logit 분포** | 마지막 layer 의 (vocab,) | "다음 토큰 후보들의 확신도" |

## 3. 어디에 쓰이나 — 5가지 표준 패턴

| 패턴 | 무엇을 보나 | 어디 |
|---|---|---|
| **Previous token** | 직전 토큰 | 모든 layer |
| **First token (BOS)** | 시퀀스 시작 | 깊은 layer |
| **Diagonal (self)** | 자기 자신 | 모든 layer |
| **Induction** | 같은 문맥 이전 위치 | 중간 layer (Anthropic 발견) |
| **Position-skip** | 일정 거리 떨어진 위치 | 표제어·반복 패턴 |

## 4. 최소 예제 — Attention map 추출

`F.scaled_dot_product_attention` 은 attention 가중치를 반환하지 않으므로 (FlashAttention 메모리 최적화), 시각화 위해 수동 구현으로 한 번 더 forward 합니다.

In [ ]:
# attn_extract.py
# from nano_gpt import GPTMini, GPTConfig, apply_rope  # Ch 10 에서 정의

# cfg = GPTConfig(...)
# model = GPTMini(cfg).to(device).eval()
# state = torch.load("runs/exp1/final.pt")
# model.load_state_dict(state['model'])

@torch.no_grad()
def get_attention(model, x):
    """모든 layer 의 (head, T, T) attention map 반환."""
    cos, sin = model.cos, model.sin
    h = model.tok_emb(x)
    maps = []
    for block in model.blocks:
        # block.attn 의 forward 를 수동 재현 -- SDPA 는 attention 가중치 반환 안 함
        attn = block.attn
        B, T, D = h.shape
        normed = block.norm1(h)
        q, k, v = attn.qkv(normed).split(D, dim=-1)
        q = q.view(B, T, attn.n_head, attn.head_dim).transpose(1, 2)
        k = k.view(B, T, attn.n_head, attn.head_dim).transpose(1, 2)
        v = v.view(B, T, attn.n_head, attn.head_dim).transpose(1, 2)
        q = apply_rope(q, cos[:T], sin[:T])
        k = apply_rope(k, cos[:T], sin[:T])

        scores = (q @ k.transpose(-2, -1)) / (attn.head_dim ** 0.5)
        mask = torch.triu(torch.ones(T, T, device=x.device), 1).bool()
        scores = scores.masked_fill(mask, float('-inf'))
        att = F.softmax(scores, dim=-1)                                 # softmax 결과 = attention map
        maps.append(att[0].cpu())                                       # (head, T, T)

        # 원래 forward 진행
        out = att @ v
        out = out.transpose(1, 2).contiguous().view(B, T, D)
        h = h + attn.proj(out)
        h = h + block.ffn(block.norm2(h))
    return maps                                                         # list of (head, T, T)

# 사용
# text = "Once upon a time, there was a little girl named"
# ids = torch.tensor([tok.encode(text).ids], device=device)
# maps = get_attention(model, ids)
# print(f"  layers: {len(maps)}, heads: {maps[0].shape[0]}")

print("get_attention 함수 정의 완료")

In [ ]:
# plot_attn.py -- 시각화
def plot_attention(maps, tokens, layer=0, head=0):
    att = maps[layer][head].numpy()
    fig, ax = plt.subplots(figsize=(8, 8))
    im = ax.imshow(att, cmap='Blues', aspect='auto')
    ax.set_xticks(range(len(tokens)))
    ax.set_xticklabels(tokens, rotation=45)
    ax.set_yticks(range(len(tokens)))
    ax.set_yticklabels(tokens)
    ax.set_xlabel("attended to")
    ax.set_ylabel("from position")
    ax.set_title(f"Layer {layer}, Head {head}")
    plt.colorbar(im)
    plt.tight_layout()
    plt.show()

# 데모: 랜덤 attention map 으로 학습 전/후 시각화
T = 8
tokens_demo = ["Once", "upon", "a", "time", ",", "there", "was", "a"]

# 인과 마스크 적용 랜덤 attention (학습 전 시뮬레이션)
torch.manual_seed(42)
raw = torch.randn(T, T)
mask = torch.triu(torch.ones(T, T), 1).bool()
raw = raw.masked_fill(mask, float('-inf'))
att_before = F.softmax(raw, dim=-1)

# 학습 후: previous token 패턴 시뮬레이션
att_after = att_before.clone()
for i in range(1, T):
    att_after[i, i-1] += 1.5
att_after = att_after.masked_fill(mask, float('-inf'))
att_after = F.softmax(att_after, dim=-1)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))
ax1.imshow(att_before.numpy(), cmap='Blues')
ax1.set_title("학습 전 (random init) -- 균등 분포")
ax1.set_xticks(range(T))
ax1.set_xticklabels(tokens_demo, rotation=45, fontsize=8)
ax1.set_yticks(range(T))
ax1.set_yticklabels(tokens_demo, fontsize=8)

ax2.imshow(att_after.numpy(), cmap='Blues')
ax2.set_title("학습 후 -- previous token 패턴 발달")
ax2.set_xticks(range(T))
ax2.set_xticklabels(tokens_demo, rotation=45, fontsize=8)
ax2.set_yticks(range(T))
ax2.set_yticklabels(tokens_demo, fontsize=8)

plt.suptitle("Attention Map 비교 (시뮬레이션)", fontsize=12)
plt.tight_layout()
plt.show()

print("본 책 모델 (학습 후) 에서 자주 보이는 패턴:")
print("  layer 0~1: 거의 자기 자신 또는 직전")
print("  layer 2~3: 일부 head 가 첫 토큰 (BOS) 에 가중")
print("  layer 4~5: 마지막 명사에 attention 집중 -- induction 시작")

## 5. 실전 — Logit 분포 추적

In [ ]:
# logit_trace.py
@torch.no_grad()
def top_k_trace(model, tok, prompt, n_steps=10, k=5):
    """다음 n 토큰 생성하면서 매 step 의 top-k 후보 출력."""
    ids = torch.tensor([tok.encode(prompt).ids], device=device)
    for step in range(n_steps):
        logits, _ = model(ids)
        probs = F.softmax(logits[0, -1], dim=-1)
        top = torch.topk(probs, k)

        print(f"\nstep {step}: prefix='{tok.decode(ids[0].tolist())}'")
        for p, i in zip(top.values.tolist(), top.indices.tolist()):
            print(f"    {tok.decode([i]):>15s}  {p:.4f}")

        # greedy 로 다음 토큰
        ids = torch.cat([ids, top.indices[:1].unsqueeze(0)], dim=1)

# 모델이 있을 때:
# top_k_trace(model, tok, "Once upon a time", n_steps=8, k=5)

print("top_k_trace 함수 정의 완료")
print()
print("전형적 출력 예시:")
print("  step 0: prefix='Once upon a time'")
print("      ,             0.6234")
print("      ,Gthere       0.1521")
print("      Gin           0.0432")
print()
print("  step 1: prefix='Once upon a time,'")
print("      Gthere        0.7821         <-- 거의 확정")
print()
print("관찰 가이드:")
print("  Top-1 > 0.7: 정형 표현, 모델이 확신")
print("  Top-5 유사 확률: 모델이 헷갈림 (인명·명사 자리)")
print("  Top-1 < 0.1: 모델이 모름, OOD 신호")

In [ ]:
# before_after.py -- 학습 전·후 attention 비교

# 학습 전
# model_init = GPTMini(cfg).to(device).eval()
# maps_before = get_attention(model_init, ids)

# 학습 후
# model_trained = GPTMini(cfg).to(device).eval()
# model_trained.load_state_dict(torch.load("runs/exp1/final.pt")['model'])
# maps_after = get_attention(model_trained, ids)

# Layer 2, head 3 비교
# fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))
# ax1.imshow(maps_before[2][3], cmap='Blues')
# ax1.set_title("학습 전 (random init)")
# ax2.imshow(maps_after[2][3], cmap='Blues')
# ax2.set_title("학습 후 (12K step)")
# plt.show()

print("학습 전: 거의 균등 (uniform) -- 마스크된 부분 빼고 모두 비슷한 색")
print("학습 후: 대각선 + 첫 토큰 (BOS) + 일부 명사에 집중")
print()
print("-> 이것이 학습이 attention 을 형성한 증거. 모델 능력의 직접적 시각화.")

## 연습문제

1. 본 책 10M 모델로 §4 의 attention 추출. layer 0, layer 5 의 head 들 중 **가장 sparse 한** (한 곳에 집중) head 를 찾아라.
2. **학습 step 1K, 5K, 12K** 의 체크포인트로 같은 prompt 의 attention 을 비교. 학습 진행에 따라 어떻게 변하나?
3. §5 의 logit trace 를 **temperature 0.0 (greedy)** vs **0.8** 로 같은 prompt 에 적용. top-1 확률이 어떻게 변화하는가?
4. 본 책 모델이 **잘못 생성한** 문장 (예: 갑작스런 화제 전환) 을 찾아 그 자리의 attention 을 분석. 어느 head 가 잘못된 자리를 보았나?
5. **(생각해볼 것)** induction head (Anthropic 발견) 가 본 책 10M 모델에서 형성되는가? 어떻게 검증할 수 있나?

In [ ]:
# 연습 1: 가장 sparse한 head 찾기


In [ ]:
# 연습 2: 학습 step별 attention 비교


In [ ]:
# 연습 3: temperature 별 logit trace


In [ ]:
# 연습 4: 실패 사례 attention 분석


In [ ]:
# 연습 5: induction head 검증
